In [71]:

import polars as pl
from datetime import date
import pandas as pd
from lifetimes import BetaGeoFitter, GammaGammaFitter
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from sklearn.metrics import root_mean_squared_error

In [72]:
df = pl.read_parquet("train.parquet")

cutoff_date = date(2026, 1, 14)
target_start = date(2026, 1, 15)
target_end = date(2026, 2, 13)

df_train = df.filter(pl.col("event_date") <= cutoff_date)

print("Data loaded successfully!")

Data loaded successfully!


In [73]:
orders = df_train.filter(pl.col("gmv") > 0)

# 2. Агрегируем исторические данные по каждому user_id
btyd_features = orders.group_by("user_id").agg(
    # Минимальная и максимальная дата покупки
    pl.col("event_date").min().alias("first_purchase"),
    pl.col("event_date").max().alias("last_purchase"),
    # Количество уникальных ДНЕЙ с покупками
    pl.col("event_date").n_unique().alias("num_orders"),
    # Средний дневной GMV за дни с покупками
    pl.col("gmv").mean().alias("monetary_value"),
)

# 3. Рассчитываем ключевые BTYD показатели
btyd_features = btyd_features.with_columns(
    # frequency = число ПОВТОРНЫХ покупок (всего покупок - 1)
    (pl.col("num_orders") - 1).alias("frequency"),
    # recency = количество дней между первой и последней покупкой
    (
        (pl.col("last_purchase") - pl.col("first_purchase")).dt.total_days()
    ).alias("recency"),
    # T (age) = количество дней от первой покупки до даты отсечки cutoff_date
    ((pl.lit(cutoff_date) - pl.col("first_purchase")).dt.total_days()).alias(
        "T"
    ),
)

# 4. Важный шаг: добавляем тех пользователей из исходного датасета, у кого ВООБЩЕ не было покупок до cutoff_date
all_users = df.select("user_id").unique()
btyd_df = all_users.join(btyd_features, on="user_id", how="left").with_columns(
    pl.col("frequency").fill_null(0),
    pl.col("recency").fill_null(0),
    pl.col("T").fill_null(0),
    pl.col("monetary_value").fill_null(0),
)

print(f"Размер итоговой BTYD-таблицы: {btyd_df.shape}")

Размер итоговой BTYD-таблицы: (250000, 8)


In [74]:
# 1. Считаем реальный таргет на периоде (берутся переменные target_start и target_end)
df_val_target = (
    df.filter(
        (pl.col("event_date") >= target_start)
        & (pl.col("event_date") <= target_end)
    )
    .group_by("user_id")
    .agg(pl.col("gmv").sum().alias("target_gmv"))
)

# 2. Объединяем BTYD-фичи с реальным таргетом
btyd_full = btyd_df.join(df_val_target, on="user_id", how="left").with_columns(
    pl.col("target_gmv").fill_null(0.0)
)

# 3. Переводим в Pandas через to_dict() (без зависимости от pyarrow)
df_p = pd.DataFrame(btyd_full.to_dict(as_series=False))

print(
    f"Подготовка целевых данных завершена для периода {target_start} — {target_end}!"
)

Подготовка целевых данных завершена для периода 2026-02-14 — 2026-03-15!


In [75]:
# 1. Обучаем BG/NBD (предсказание количества дней с покупками)
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(df_p["frequency"], df_p["recency"], df_p["T"])

# Предсказываем ожидаемое количество дней с покупками на следующие 30 дней (t=30)
df_p["pred_orders_30d"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    30, df_p["frequency"], df_p["recency"], df_p["T"]
)
df_p["pred_orders_30d"] = df_p["pred_orders_30d"].fillna(0.0).clip(lower=0.0)

# 2. Обучаем Gamma-Gamma (предсказание среднего чека)
# КРИТИЧЕСКИЙ ФИКС: Обучение строго на клиентах с frequency > 0 И monetary_value > 0
returning_customers = df_p[
    (df_p["frequency"] > 0) & (df_p["monetary_value"] > 0)
]

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    returning_customers["frequency"], returning_customers["monetary_value"]
)

# Предсказываем ожидаемый средний чек
df_p["pred_monetary"] = ggf.conditional_expected_average_profit(
    df_p["frequency"], df_p["monetary_value"]
)

# Заполняем пропуски (для тех, у кого frequency == 0) средним значением чека среди покупавших
mean_monetary_val = returning_customers["monetary_value"].mean()
df_p["pred_monetary"] = df_p["pred_monetary"].fillna(mean_monetary_val)

# 3. Итоговое предсказание GMV за 30 дней:
df_p["predict"] = df_p["pred_orders_30d"] * df_p["pred_monetary"]

# Финальная зачистка отрицательных и NaN значений
df_p["predict"] = df_p["predict"].fillna(0.0).clip(lower=0.0)

print("Модели успешно обучены!")

Модели успешно обучены!


In [76]:
# Формула RMSLE
y_true_log = np.log1p(df_p["target_gmv"])
y_pred_log = np.log1p(df_p["predict"])

val_rmsle = root_mean_squared_error(y_true_log, y_pred_log)
print(f"Validation RMSLE Score: {val_rmsle:.5f}")

Validation RMSLE Score: 3.75166


In [77]:
# 1. Формируем сабмит-датасет с колонками user_id и gmv (predict)
sub_df = df_p[["user_id", "predict"]].rename(columns={"predict": "gmv"}).copy()


output_filename = f"competition_test_{cutoff_date}.csv"
sub_df.to_csv(output_filename, index=False)

with open(output_filename, "a", encoding="utf-8") as f:
    f.write(
        f"\n# Validation RMSLE Score for cutoff {cutoff_date}: {val_rmsle:.5f}\n"
    )

print(f"Файл успешно сохранен: {output_filename}")
print(f"Всего пользователей в файле: {len(sub_df)}")
print(f"Зафиксированное значение RMSLE: {val_rmsle:.5f}")

sub_df.head()

Файл успешно сохранен: competition_test_2026-02-13.csv
Всего пользователей в файле: 250000
Зафиксированное значение RMSLE: 3.75166


,user_id,gmv
0,836567,-0.000000
1,287728,6.183315
2,607182,-0.000000
3,525899,25.923635
4,756231,52.298626
